# Customer Churn Analysis — Telecom EDA

**Author:** Vinay  
**Dataset:** Telecom Customer Churn (Training: ~440K rows | Test: ~64K rows)  
**Goal:** Identify which customer segments are most at risk of churning, and which behavioural signals are the strongest predictors of churn.

---

## Business Context

Customer churn is one of the most costly problems a telecom company faces. Acquiring a new customer is typically 5–7× more expensive than retaining an existing one. This analysis explores a real-world telecom dataset to surface the key drivers of churn and provide actionable insights for retention strategy.

**Key questions we'll answer:**
1. How severe is the churn problem? What fraction of customers are leaving?
2. Are there demographic patterns — does age or gender correlate with churn?
3. Do contract type and subscription tier significantly affect churn rates?
4. Which numeric features (payment delay, support calls, tenure, spend) differ most between churned and retained customers?
5. How do these features correlate with each other?

---

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Loading & First Look](#2-data-loading--first-look)
3. [Data Cleaning](#3-data-cleaning)
4. [Univariate Analysis](#4-univariate-analysis)
5. [Bivariate Analysis — Churn vs Features](#5-bivariate-analysis--churn-vs-features)
6. [Correlation Heatmap](#6-correlation-heatmap)
7. [Key Insights & Conclusions](#7-key-insights--conclusions)


## 1. Environment Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')

ACCENT_BLUE = 'royalblue'
ACCENT_RED = 'indianred'
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

## 2. Data Loading and First Look

In [10]:
train = pd.read_csv(
    r"C:\Users\vinay\Desktop\Customer Churn Data\Data\customer_churn_dataset-training-master.csv"
)

test = pd.read_csv(
    r"C:\Users\vinay\Desktop\Customer Churn Data\Data\customer_churn_dataset-testing-master.csv"
)

In [11]:
train.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [12]:
test.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [13]:
train.shape

(440833, 12)

In [14]:
test.shape

(64374, 12)

In [15]:
train.columns

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Churn'],
      dtype='object')

In [16]:
test.columns

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Churn'],
      dtype='object')

In [17]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


In [ ]:
test.info()

In [ ]:
# Descriptive statistics for the training set
train.describe().round(2)

* 75% customers have tenure less than 46 months

* Average total spending is 631.61 

In [ ]:
test.describe().round(2)

* Here average tenure is less than 31.99 months

* The average payment delay is 17.1 days with the median(50%) at 19 days

## 3. Data Cleaning

Before exploring, we tidy up three things:
1. **Check for missing values** and decide how to handle them.
2. **Create a Tenure Group** feature to make tenure easier to reason about.
3. **Fix data types** — the training set loaded floats where integers are appropriate.

#### Missing Value Treatment

In [ ]:
# Visualize missing values
missing = (
    train.isnull()
         .sum()
         .mul(100)
         .div(len(train))
         .reset_index()
)

missing.columns = [
    'Column',
    'Missing Percentage'
]
plt.figure(figsize=(12,5))

sns.pointplot(
    data=missing,
    x='Column',
    y='Missing Percentage',
    color = ACCENT_BLUE 
)

plt.xticks(rotation=45)
plt.title('Percentage of Missing Values')
plt.ylabel('Missing Percentage (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.show()

* For features with less missing values- can use regression to predict the missing values or fill with the mean of the values present, depending on the feature.
* For features with very high number of missing values- it is better to drop those columns as they give very less insight on analysis.

In [ ]:
test.isnull().sum()

In [ ]:
print('Missing values per column (training set):')
train.isnull().sum()

* Since the % of these records compared to total dataset is very low, it is safe to drop them from further processing.

In [ ]:
#Removing missing values
train.dropna(how = 'any', inplace = True)

In [ ]:
# Bin Tenure into human-readable yearly groups
bins = [0, 12, 24, 36, 48, 60, 72]

labels = [
    '0-1 Yr',
    '1-2 Yrs',
    '2-3 Yrs',
    '3-4 Yrs',
    '4-5 Yrs',
    '5-6 Yrs'
]

for df in [train, test]:
    df['Tenure Group'] = pd.cut(
        df['Tenure'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

In [ ]:
train['Tenure Group'].value_counts().sort_index()

In [ ]:
test['Tenure Group'].value_counts().sort_index()

#### Data Type Conversion

In [ ]:
#Fix dtypes: float → int for integer columns
for col in train.select_dtypes(include='float'): 
    train[col] = train[col].astype('int64')
print('Data types after conversion:')
print(train.dtypes)

## 4. Univariate Analysis
We first look at each variable in isolation to understand the shape and distribution of the data.

In [ ]:
# Churn rate overview
churn_counts = train['Churn'].value_counts()
churn_pct    = train['Churn'].mean() * 100

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    churn_counts,
    labels=['Churned', 'Retained'],
    colors=[ACCENT_BLUE,ACCENT_RED],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
ax.set_title(f'Overall Churn Rate: {churn_pct:.1f}%', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Churned  : {churn_counts[1]:,} ({churn_pct:.1f}%)')
print(f'Retained : {churn_counts[0]:,} ({100 - churn_pct:.1f}%)')

In [ ]:
# Age Distribution 
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(train['Age'], bins=30, kde=True, color=ACCENT_BLUE, ax=ax)
ax.set_title('Customer Age Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

# Most customers fall in the 30-40 (middle-aged) bracket - important for targeting 
print(train['Age'].describe().round(1))

In [ ]:
# Categorical feature distributions 
cat_cols = ['Gender', 'Subscription Type', 'Contract Length']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(cat_cols):
     order = train[col].value_counts().index
     sns.countplot(
       data=train,
        x=col,
        order=order,
        palette = 'Blues_d',
        ax=axes[i]
    )
    
     axes[i].set_title(col)
     axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Bivariate Analysis - Churn vs Features

In [ ]:
# Churn rate by Contract Length

plt.figure(figsize=(8,5))
contract_churn = (
    train.groupby('Contract Length')['Churn']
    .mean() * 100
).sort_values(ascending=False)

bars = plt.bar(
    contract_churn.index,
    contract_churn.values,
    color=[ACCENT_BLUE if x > 50 else ACCENT_RED
           for x in contract_churn]
)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{bar.get_height():.1f}%',
        ha='center', va='bottom', fontweight='bold'
    )

plt.axhline(50, ls='--', c='grey', label=('50% Reference'))
plt.title('Churn Rate by Contract Length')
plt.xlabel('Contract Length')
plt.ylabel('Churn Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by Subscription Type
plt.figure(figsize=(8,5))

sub_churn = (train.groupby('Subscription Type')['Churn']
            .mean()*100
).sort_values(ascending = False)

bars = plt.bar(sub_churn.index,
              sub_churn.values, 
              color=[ACCENT_BLUE if v > 50 else ACCENT_RED
                     for v in sub_churn]
)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2, 
        bar.get_height() + 0.5, 
        f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold'
)

plt.title('Churn by Subscription Type')
plt.xlabel('Subscription Type')
plt.ylabel('Churn Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by Tenure Group
plt.figure(figsize=(9, 5)) 

tenure_churn = (train.groupby('Tenure Group', observed=True)['Churn']
                .mean()
                .mul(100)
)

tenure_bars = plt.bar(
    tenure_churn.index, 
    tenure_churn.values,
    color= [ACCENT_BLUE if v > 50 else ACCENT_RED for v in tenure_churn], 
    alpha = 0.9
)

for b in tenure_bars:
    plt.text(
        b.get_x() + b.get_width() / 2, 
        b.get_height() + 0.5, 
        f'{b.get_height():.1f}%', 
        ha= 'center', va='bottom', fontweight='bold'
)

plt.axhline(50, linestyle='--', color='grey', label='50% reference')
plt.title('Churn Rate by Tenure Group')
plt.xlabel('Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter()) 
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by gender
plt.figure(figsize=(8,5))

gen_churn = (train.groupby('Gender')['Churn']
            .mean()*100
).sort_values(ascending = False)

gbars = plt.bar(gen_churn.index,
              gen_churn.values, 
              color=[ACCENT_BLUE if v > 50 else ACCENT_RED
                     for v in gen_churn]
)

for b in gbars:
    plt.text(
        b.get_x() + b.get_width()/2, 
        b.get_height() + 0.5, 
        f'{b.get_height():.1f}%', ha='center', va='bottom', fontweight='bold'
)

plt.axhline(50, ls='--', c='grey', label='50% Threshold')
plt.title('Churn by Gender')
plt.xlabel('Gender')
plt.ylabel('Churn Rate (%)')
plt.legend()
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

In [ ]:
# Create age bins
age_bins   = [18, 25, 35, 45, 55, 65, 100]
age_labels = ['18–25', '26–35', '36–45', '46–55', '56–65', '65+']
train['Age Group'] = pd.cut(
    train['Age'], bins=age_bins, labels=age_labels, include_lowest=True
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — KDE by churn status
sns.kdeplot(
    data=train, x='Age', hue='Churn',
    palette=(ACCENT_RED, ACCENT_BLUE),
    fill=True, alpha=0.3, linewidth=2, ax=axes[0]
)
axes[0].set_title('Age Distribution by Churn Status', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Density')

# Right — churn rate by age group
age_churn = (
    train.groupby('Age Group', observed=True)['Churn']
    .mean() * 100
)
abars = axes[1].bar(
    age_churn.index, age_churn.values,
    color=[ACCENT_BLUE if v > 50 else ACCENT_RED for v in age_churn],
    alpha=0.85
)
for b in abars:
    axes[1].text(
        b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
        f'{b.get_height():.1f}%', ha='center', va='bottom', fontweight='bold'
    )
axes[1].axhline(50, ls='--', c='grey', linewidth=1.2)
axes[1].set_title('Churn Rate by Age Group', fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_ylim(0, 100)

plt.suptitle('Age vs Churn', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: continuous features by churn status
# These reveal whether churned customers differ on spend, delay, calls, etc.
cont_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
             'Payment Delay', 'Total Spend', 'Last Interaction']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

churn_labels = {0: 'Retained', 1: 'Churned'}
train['Churn Label'] = train['Churn'].map(churn_labels)

for i, col in enumerate(cont_cols):
    sns.boxplot(
        data=train, x='Churn Label', y=col,
        palette={'Retained': ACCENT_RED, 'Churned': ACCENT_BLUE},
        order=['Retained', 'Churned'],
        ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('')

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Distribution of Numeric Features by Churn Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap
A correlation matrix across all numeric features (including the target Churn) tells us which features move together — and which are most linearly related to churn.

In [ ]:
num_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
            'Payment Delay', 'Total Spend', 'Last Interaction', 'Churn'] 
corr = train[num_cols].corr()
fig, ax = plt.subplots(figsize=(10,8))

sns.heatmap(corr,
    annot=True,
    cmap='coolwarm'
)
ax.set_title('Correlation Matrix - Numeric Features (incl. Churn)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Features ranked by correlation with Churn ──────────────────────────────
churn_corr = (
    corr['Churn']
    .drop('Churn')
    .sort_values(key=abs, ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 6))
colors = [ACCENT_RED if v > 0 else ACCENT_BLUE for v in churn_corr]
ax.barh(churn_corr.index, churn_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Churn (sorted by |correlation|)')
ax.set_xlabel('Pearson Correlation Coefficient')
for i, v in enumerate(churn_corr.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Key Insights & Conclusions

Below are the top findings from this analysis, framed as business-actionable insights.

---

### 1. Churn is a serious problem
The overall churn rate is **~56.7%** in the training set — well above the industry average. More than half of customers are leaving, which represents a major revenue risk.

---

### 2. Contract type is the single strongest churn predictor
- **Monthly contract** customers churn at roughly **2× the rate** of annual contract customers.
- This suggests that locking customers into longer contracts is an effective retention lever — but care must be taken not to alienate customers who aren't ready to commit.

---

### 3. Support call volume is a reliable early-warning signal
- Churned customers make significantly **more support calls** than retained customers (visible in both the box plots and the correlation matrix).
- A spike in support calls likely signals unresolved frustration — intervening proactively after a certain number of calls could stem churn.

---

### 4 Payment delay is the second-strongest numeric predictor
- Churned customers show noticeably **higher payment delays**.
- A delay threshold (e.g. >15 days) could serve as a trigger for outreach — a discount or payment plan offer might recover these customers before they leave.

---

### 5 Tenure has a non-linear relationship with churn
- Churn risk is highest in the **0–1 year** cohort, drops for 2–4 year customers, then rises again for long-tenured customers.
- The first year is critical — strong onboarding and early-life engagement are essential.

---

### 6 Total Spend and Usage Frequency are weakly correlated with churn
- Higher spenders do **not** necessarily churn less — suggesting that price point is not the primary driver of retention.
- Focus should be on service quality and contract type rather than discounting.

---

### 7 Gender and subscription tier show minimal signal
- The Gender split is nearly 50/50 and shows little difference in churn rates.
- All three subscription tiers (Basic, Standard, Premium) have relatively balanced distributions and similar churn patterns.